In [1]:
# CLARiTy-style anatomy-gated analysis for ALBEF heatmaps on VinDr-CXR with FROC

# What this notebook does
# ---------------------
# 1. Loads saved ALBEF heatmaps for VinDr test images using crossattn_gradcam_index.csv.
# 2. Restricts analysis to two labels:
#       - Cardiomegaly
#       - Pleural effusion
# 3. Samples 50-100 positive VinDr test cases per label.
# 4. For each sampled case, saves a panel showing:
#       - original image
#       - raw ALBEF heatmap
#       - anatomy mask
#       - refined heatmap
#       - GT boxes + raw predicted boxes
#       - GT boxes + anatomy-gated predicted boxes
# 5. Computes a simple diagnostic:
#       - fraction of raw heatmap mass inside the heart mask for Cardiomegaly
#       - fraction of raw heatmap mass inside the lung/lower-lung mask for Pleural effusion
# 6. Builds raw and anatomy-gated predictions.
# 7. Runs FROC internally for both raw and anatomy-gated pipelines.
# 8. Saves summaries and per-label FROC curves.

In [2]:
import argparse
import json
import os
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.patches import Rectangle
from PIL import Image
from scipy import ndimage as ndi
import torchxrayvision as xrv

/home/hpc/iwi5/iwi5362h/.local/lib/python3.9/site-packages/torch/torch_version.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]


In [3]:
# ============================================================
# Configuration
# ============================================================
TARGET_LABELS = ["Cardiomegaly", "Pleural effusion"]

LABEL_TO_ANATOMY = {
    "Cardiomegaly": "heart",
    "Pleural effusion": "effusion",
}

DEFAULT_THRESHOLDS = {
    "Cardiomegaly": 0.50,
    "Pleural effusion": 0.45,
}

DEFAULT_MIN_BOX_AREA_FRAC = {
    "Cardiomegaly": 0.002,
    "Pleural effusion": 0.001,
}

HEATMAP_KEY_MAP = {
    "Cardiomegaly": "Cardiomegaly",
    "Pleural effusion": "Pleural effusion",
}

heatmaps_dir = "/home/woody/iwi5/iwi5362h/ALBEF/results/zero_shot_vindr_results/heatmaps_vindr_raw_vis"
annotations_csv = "/home/woody/iwi5/iwi5362h/data/vindr_cxr/annotations/annotations_test.csv"
meta_csv = "/home/woody/iwi5/iwi5362h/data/vindr_cxr/test_meta.csv"
images_root = "/home/woody/iwi5/iwi5362h/data/vindr_cxr/test"
output_dir = "/home/woody/iwi5/iwi5362h/ALBEF/results/clarity_end_to_end"
samples_per_label = 75
seed = 42
device = "cuda"

In [4]:
@dataclass
class Box:
    x1: float
    y1: float
    x2: float
    y2: float
    score: float
    label: str
    image_id: str


# ============================================================
# General utilities
# ============================================================
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def safe_filename(name: str) -> str:
    name = str(name).replace("/", "_").replace("\\", "_")
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name if name else "label"


def minmax_norm(x: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    xmin = float(np.nanmin(x))
    xmax = float(np.nanmax(x))
    if xmax - xmin < eps:
        return np.zeros_like(x, dtype=np.float32)
    return (x - xmin) / (xmax - xmin + eps)


def resize_map(arr: np.ndarray, out_hw: Tuple[int, int], interpolation=cv2.INTER_LINEAR) -> np.ndarray:
    out_h, out_w = out_hw
    return cv2.resize(arr.astype(np.float32), (out_w, out_h), interpolation=interpolation)


# ============================================================
# CSV loading
# ============================================================
def load_annotations(annotations_csv: Path) -> pd.DataFrame:
    ann = pd.read_csv(annotations_csv)
    ann.columns = [c.strip() for c in ann.columns]
    required = {"image_id", "class_name", "x_min", "y_min", "x_max", "y_max"}
    missing = required - set(ann.columns)
    if missing:
        raise ValueError(f"annotations_csv missing columns: {sorted(missing)}")
    return ann


def load_meta(meta_csv: Path) -> pd.DataFrame:
    meta = pd.read_csv(meta_csv)
    meta.columns = [c.strip() for c in meta.columns]
    width_col = next((c for c in ["width", "W", "image_width", "dim1"] if c in meta.columns), None)
    height_col = next((c for c in ["height", "H", "image_height", "dim0"] if c in meta.columns), None)
    if width_col is None or height_col is None or "image_id" not in meta.columns:
        raise ValueError(
            "meta_csv must contain image_id and width/height columns. "
            f"Found columns: {list(meta.columns)}"
        )
    meta = meta.rename(columns={width_col: "width", height_col: "height"})
    return meta[["image_id", "width", "height"]].copy()


def load_heatmap_index(heatmaps_dir: Path) -> pd.DataFrame:
    idx_csv = heatmaps_dir / "crossattn_gradcam_index.csv"
    if not idx_csv.exists():
        raise FileNotFoundError(f"Missing heatmap index CSV: {idx_csv}")

    idx = pd.read_csv(idx_csv)
    idx.columns = [c.strip() for c in idx.columns]

    img_col = next((c for c in ["image_id", "dicom_id", "id"] if c in idx.columns), None)
    path_col = next((c for c in ["heatmap_path", "path", "file", "pt_path", "npz_path"] if c in idx.columns), None)

    if img_col is None or path_col is None:
        raise ValueError(
            "crossattn_gradcam_index.csv must contain image_id-like and path-like columns. "
            f"Found columns: {list(idx.columns)}"
        )

    idx = idx.rename(columns={img_col: "image_id", path_col: "heatmap_path"})

    def _resolve(p: str) -> str:
        p = str(p)
        return str((heatmaps_dir / p).resolve()) if not os.path.isabs(p) else p

    idx["heatmap_path"] = idx["heatmap_path"].astype(str).map(_resolve)
    return idx[["image_id", "heatmap_path"]].drop_duplicates().copy()


# ============================================================
# Image loading
# ============================================================
def load_xray_image(path: Path) -> np.ndarray:
    img = Image.open(path).convert("L")
    arr = np.asarray(img)
    if arr.dtype != np.uint8:
        arr = (minmax_norm(arr) * 255.0).astype(np.uint8)
    return arr


# ============================================================
# Heatmap loading
# ============================================================
def _to_python_obj(obj):
    if isinstance(obj, np.ndarray) and obj.dtype == object and obj.size == 1:
        return obj.item()
    return obj


def load_heatmap_file(path: str):
    path = str(path)
    if path.endswith(".pt") or path.endswith(".pth"):
        return torch.load(path, map_location="cpu")
    if path.endswith(".npz"):
        data = np.load(path, allow_pickle=True)
        if "heatmaps" in data.files:
            return _to_python_obj(data["heatmaps"])
        if len(data.files) == 1:
            return _to_python_obj(data[data.files[0]])
        return {k: _to_python_obj(data[k]) for k in data.files}
    if path.endswith(".npy"):
        return _to_python_obj(np.load(path, allow_pickle=True))
    raise ValueError(f"Unsupported heatmap file extension: {path}")


def extract_cam_vis(heatmaps_obj, pathology_key: str) -> np.ndarray:
    cam = heatmaps_obj[pathology_key]["cam_vis"]
    cam = np.asarray(cam, dtype=np.float32)
    cam = np.squeeze(cam)
    if cam.ndim != 2:
        raise ValueError(f"Expected 2D cam_vis, got shape {cam.shape}")
    return cam


def load_heatmap_for_image(heatmap_path: str, label: str) -> np.ndarray:
    obj = load_heatmap_file(heatmap_path)
    pathology_key = HEATMAP_KEY_MAP[label]
    return extract_cam_vis(obj, pathology_key)


# ============================================================
# Anatomy segmentation
# ============================================================
class AnatomySegmenter:
    """
    TorchXRayVision ChestX-Det PSPNet wrapper.
    Compatible with older torch versions that do not support
    interpolate(..., antialias=True).
    """

    def __init__(self, device: str = "cpu"):
        self.xrv = xrv
        self.device = device
        self.model = xrv.baseline_models.chestx_det.PSPNet().to(device)
        self.model.eval()
        self.targets = list(self.model.targets)

        self.idx_heart = self.targets.index("Heart")
        self.idx_left_lung = self.targets.index("Left Lung")
        self.idx_right_lung = self.targets.index("Right Lung")

    def _prep(self, img_u8: np.ndarray) -> torch.Tensor:
        # img_u8: H x W grayscale uint8
        x = img_u8.astype(np.float32)

        # Normalize using xrv helper
        x = self.xrv.datasets.normalize(x, 255)  # -> roughly [-1024, 1024]

        # Manually resize to 512x512 BEFORE giving to the model,
        # so xrv does not call fix_resolution(..., antialias=True)
        x = cv2.resize(x, (512, 512), interpolation=cv2.INTER_LINEAR)

        # shape: [1, 1, 512, 512]
        x = torch.from_numpy(x).unsqueeze(0).unsqueeze(0).to(self.device)
        return x

    @torch.no_grad()
    def segment(self, img_u8: np.ndarray) -> Dict[str, np.ndarray]:
        x = self._prep(img_u8)

        # PSPNet.forward repeats channel internally, so input can stay 1-channel
        out = self.model(x)  # [1, C, 512, 512]
        out = torch.sigmoid(out)[0].detach().cpu().numpy()

        h, w = img_u8.shape[:2]
        heart = resize_map(out[self.idx_heart], (h, w), interpolation=cv2.INTER_LINEAR)
        left_lung = resize_map(out[self.idx_left_lung], (h, w), interpolation=cv2.INTER_LINEAR)
        right_lung = resize_map(out[self.idx_right_lung], (h, w), interpolation=cv2.INTER_LINEAR)
        lungs = np.clip(left_lung + right_lung, 0.0, 1.0)

        return {
            "heart": minmax_norm(heart),
            "left_lung": minmax_norm(left_lung),
            "right_lung": minmax_norm(right_lung),
            "lungs": minmax_norm(lungs),
        }


def build_effusion_mask(lungs_mask: np.ndarray, lower_weight_power: float = 1.5) -> np.ndarray:
    h, w = lungs_mask.shape
    yy = np.linspace(0.0, 1.0, h, dtype=np.float32)[:, None]
    lower_weight = np.power(yy, lower_weight_power)
    lower_weight = np.repeat(lower_weight, w, axis=1)
    eff_mask = lungs_mask * lower_weight
    return minmax_norm(eff_mask)


def get_anatomy_mask(label: str, anatomy: Dict[str, np.ndarray]) -> np.ndarray:
    anatomy_type = LABEL_TO_ANATOMY[label]
    if anatomy_type == "heart":
        return anatomy["heart"]
    if anatomy_type == "effusion":
        return build_effusion_mask(anatomy["lungs"])
    raise ValueError(f"Unknown anatomy mapping for label={label}")


# ============================================================
# Heatmap refinement and diagnostics
# ============================================================
def refine_heatmap(raw_heatmap: np.ndarray, anatomy_mask: np.ndarray, label: str) -> np.ndarray:
    if label == "Cardiomegaly":
        refined = raw_heatmap * anatomy_mask
    elif label == "Pleural effusion":
        alpha = 0.2
        refined = raw_heatmap * (alpha + (1.0 - alpha) * anatomy_mask)
    else:
        refined = raw_heatmap * anatomy_mask
    return minmax_norm(refined)


def threshold_heatmap(heatmap: np.ndarray, label: str) -> np.ndarray:
    thr = DEFAULT_THRESHOLDS[label]
    return (heatmap >= float(thr)).astype(np.uint8)


def heatmap_mass_inside_mask(raw_heatmap: np.ndarray, anatomy_mask: np.ndarray, eps: float = 1e-8) -> float:
    heat = np.clip(raw_heatmap.astype(np.float32), 0.0, None)
    numerator = float((heat * anatomy_mask).sum())
    denominator = float(heat.sum()) + eps
    return numerator / denominator


# ============================================================
# GT boxes
# ============================================================
def get_gt_boxes(ann_df: pd.DataFrame, image_id: str, label: str) -> List[Box]:
    sub = ann_df[(ann_df["image_id"] == image_id) & (ann_df["class_name"] == label)]
    boxes = []
    for _, r in sub.iterrows():
        boxes.append(Box(float(r.x_min), float(r.y_min), float(r.x_max), float(r.y_max), 1.0, label, image_id))
    return boxes


def boxes_to_df(boxes: Sequence[Box]) -> pd.DataFrame:
    rows = []
    for b in boxes:
        rows.append({
            "image_id": b.image_id,
            "class_name": b.label,
            "x_min": b.x1,
            "y_min": b.y1,
            "x_max": b.x2,
            "y_max": b.y2,
            "score": b.score,
        })
    return pd.DataFrame(rows)


# ============================================================
# CAM -> boxes
# ============================================================
def heatmap_to_boxes(
    binary_map: np.ndarray,
    score_map: np.ndarray,
    image_id: str,
    label: str,
    connectivity: int = 2,
) -> List[Box]:
    h, w = binary_map.shape
    min_area_frac = DEFAULT_MIN_BOX_AREA_FRAC[label]
    min_area = max(1, int(round(min_area_frac * h * w)))

    num_labels, labeled, stats, _ = cv2.connectedComponentsWithStats(
        binary_map.astype(np.uint8), connectivity=8 if connectivity == 2 else 4
    )

    boxes: List[Box] = []
    for comp_id in range(1, num_labels):
        x, y, bw, bh, area = stats[comp_id]
        if area < min_area:
            continue

        x1, y1, x2, y2 = int(x), int(y), int(x + bw), int(y + bh)
        score = float(score_map[y1:y2, x1:x2].max()) if (y2 > y1 and x2 > x1) else 0.0
        boxes.append(Box(x1, y1, x2, y2, score, label, image_id))

    boxes.sort(key=lambda b: b.score, reverse=True)
    return boxes


# ============================================================
# Visualization
# ============================================================
def draw_boxes(ax, boxes: Sequence[Box], color: str, linewidth: float = 2.0, linestyle: str = "-") -> None:
    for b in boxes:
        rect = Rectangle(
            (b.x1, b.y1),
            b.x2 - b.x1,
            b.y2 - b.y1,
            fill=False,
            edgecolor=color,
            linewidth=linewidth,
            linestyle=linestyle,
        )
        ax.add_patch(rect)


def save_case_panel(
    save_path: Path,
    image: np.ndarray,
    raw_heatmap: np.ndarray,
    anatomy_mask: np.ndarray,
    refined_heatmap: np.ndarray,
    gt_boxes: Sequence[Box],
    pred_boxes_raw: Sequence[Box],
    pred_boxes_ref: Sequence[Box],
    title: str,
) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.ravel()

    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("Original image")
    axes[0].axis("off")

    axes[1].imshow(image, cmap="gray")
    axes[1].imshow(raw_heatmap, cmap="jet", alpha=0.45)
    axes[1].set_title("Raw ALBEF heatmap")
    axes[1].axis("off")

    axes[2].imshow(anatomy_mask, cmap="viridis")
    axes[2].set_title("Anatomy mask")
    axes[2].axis("off")

    axes[3].imshow(image, cmap="gray")
    axes[3].imshow(refined_heatmap, cmap="jet", alpha=0.45)
    axes[3].set_title("Refined heatmap")
    axes[3].axis("off")

    axes[4].imshow(image, cmap="gray")
    draw_boxes(axes[4], gt_boxes, color="lime", linewidth=2.5)
    draw_boxes(axes[4], pred_boxes_raw, color="red", linewidth=1.5)
    axes[4].set_title("Raw pipeline: GT (green) + pred (red)")
    axes[4].axis("off")

    axes[5].imshow(image, cmap="gray")
    draw_boxes(axes[5], gt_boxes, color="lime", linewidth=2.5)
    draw_boxes(axes[5], pred_boxes_ref, color="cyan", linewidth=1.5)
    axes[5].set_title("Anatomy-gated: GT (green) + pred (cyan)")
    axes[5].axis("off")

    fig.suptitle(title)
    fig.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# FROC implementation (from your existing logic)
# ============================================================
def quadrant_of_point(x: float, y: float, center_x: float, center_y: float) -> int:
    if x < center_x and y < center_y:
        return 0  # TL
    if x >= center_x and y < center_y:
        return 1  # TR
    if x < center_x and y >= center_y:
        return 2  # BL
    return 3      # BR


def box_center_xy(box_xyxy: Tuple[float, float, float, float]) -> Tuple[float, float]:
    x0, y0, x1, y1 = box_xyxy
    return (x0 + x1) / 2.0, (y0 + y1) / 2.0


def quadrant_match(pred_box: Tuple[float, float, float, float],
                   gt_box: Tuple[float, float, float, float],
                   image_w: float,
                   image_h: float) -> bool:
    px, py = box_center_xy(pred_box)
    gx, gy = box_center_xy(gt_box)
    center_x = image_w / 2.0
    center_y = image_h / 2.0
    return quadrant_of_point(px, py, center_x, center_y) == quadrant_of_point(gx, gy, center_x, center_y)


def evaluate_froc_for_label(
    predictions: Sequence[Box],
    gt_boxes: Sequence[Box],
    label: str,
    num_images: int,
    image_size_lookup: Dict[str, Tuple[int, int]],
) -> Tuple[List[Tuple[float, float]], Dict[float, float]]:
    preds = [p for p in predictions if p.label == label]
    gts = [g for g in gt_boxes if g.label == label]

    if len(gts) == 0:
        return [], {0.10: 0.0, 0.25: 0.0, 0.50: 0.0}

    gt_by_image: Dict[str, List[Box]] = {}
    for g in gts:
        gt_by_image.setdefault(g.image_id, []).append(g)

    preds = sorted(preds, key=lambda x: -x.score)

    matched_gt = set()
    tp = 0
    fp = 0
    curve: List[Tuple[float, float]] = []

    for pred in preds:
        img_id = pred.image_id
        pbox = (pred.x1, pred.y1, pred.x2, pred.y2)
        image_w, image_h = image_size_lookup[img_id]

        matched = False
        for g in gt_by_image.get(img_id, []):
            gt_key = (img_id, label, g.x1, g.y1, g.x2, g.y2)
            if gt_key in matched_gt:
                continue
            gbox = (g.x1, g.y1, g.x2, g.y2)
            if quadrant_match(pbox, gbox, image_w=image_w, image_h=image_h):
                matched_gt.add(gt_key)
                matched = True
                break

        if matched:
            tp += 1
        else:
            fp += 1

        fp_per_image = fp / float(num_images)
        sensitivity = tp / float(len(gts))
        curve.append((fp_per_image, sensitivity))

    targets = [0.10, 0.25, 0.50]
    sens_at = {}
    for t in targets:
        valid = [s for f, s in curve if f <= t]
        sens_at[t] = max(valid) if valid else 0.0

    return curve, sens_at


def evaluate_froc(
    labels: Sequence[str],
    predictions: Sequence[Box],
    gt_boxes: Sequence[Box],
    num_images: int,
    image_size_lookup: Dict[str, Tuple[int, int]],
    output_dir: Path,
    prefix: str,
) -> pd.DataFrame:
    rows = []
    output_dir.mkdir(parents=True, exist_ok=True)

    for label in labels:
        curve, sens_at = evaluate_froc_for_label(
            predictions=predictions,
            gt_boxes=gt_boxes,
            label=label,
            num_images=num_images,
            image_size_lookup=image_size_lookup,
        )

        n_gt = sum(1 for g in gt_boxes if g.label == label)
        n_pred = sum(1 for p in predictions if p.label == label)

        curve_path = output_dir / f"{prefix}_froc_curve_{safe_filename(label)}.csv"
        pd.DataFrame(curve, columns=["fp_per_image", "sensitivity"]).to_csv(curve_path, index=False)

        rows.append({
            "label": label,
            "sens@0.10": sens_at[0.10],
            "sens@0.25": sens_at[0.25],
            "sens@0.50": sens_at[0.50],
            "num_gt_boxes": n_gt,
            "num_preds": n_pred,
            "curve_csv": str(curve_path),
        })

        print(
            f"[FROC:{prefix}] {label:20s} GT={n_gt:4d} Pred={n_pred:5d} "
            f"S@0.10={sens_at[0.10]:.4f} S@0.25={sens_at[0.25]:.4f} S@0.50={sens_at[0.50]:.4f}"
        )

    out_df = pd.DataFrame(rows).sort_values(by="label")
    out_df.to_csv(output_dir / f"{prefix}_froc_summary.csv", index=False)
    return out_df


# ============================================================
# Main per-case processing
# ============================================================
def process_one_case(
    image_id: str,
    label: str,
    heatmap_path: str,
    images_root: str,
    ann_df: pd.DataFrame,
    segmenter: AnatomySegmenter,
    save_visuals: bool,
    vis_dir: Path,
) -> Tuple[List[Box], List[Box], Dict[str, object]]:
    img_path = Path(images_root) / f"{image_id}.png"
    image = load_xray_image(img_path)
    H, W = image.shape[:2]

    raw_hm_small = load_heatmap_for_image(heatmap_path, label)
    raw_heatmap = resize_map(minmax_norm(raw_hm_small), (H, W), interpolation=cv2.INTER_LINEAR)
    raw_heatmap = minmax_norm(raw_heatmap)

    anatomy = segmenter.segment(image)
    anatomy_mask = get_anatomy_mask(label, anatomy)
    refined_heatmap = refine_heatmap(raw_heatmap, anatomy_mask, label)

    mass_inside = heatmap_mass_inside_mask(raw_heatmap, anatomy_mask)
    gt_boxes = get_gt_boxes(ann_df, image_id, label)

    raw_bin = threshold_heatmap(raw_heatmap, label)
    ref_bin = threshold_heatmap(refined_heatmap, label)

    pred_boxes_raw = heatmap_to_boxes(raw_bin, raw_heatmap, image_id, label)
    pred_boxes_ref = heatmap_to_boxes(ref_bin, refined_heatmap, image_id, label)

    if save_visuals:
        title = f"{label} | {image_id} | mass_inside={mass_inside:.3f}"
        save_case_panel(
            save_path=vis_dir / f"{image_id}.png",
            image=image,
            raw_heatmap=raw_heatmap,
            anatomy_mask=anatomy_mask,
            refined_heatmap=refined_heatmap,
            gt_boxes=gt_boxes,
            pred_boxes_raw=pred_boxes_raw,
            pred_boxes_ref=pred_boxes_ref,
            title=title,
        )

    diag = {
        "label": label,
        "image_id": image_id,
        "heatmap_path": heatmap_path,
        "image_path": str(img_path),
        "height": H,
        "width": W,
        "mass_inside_anatomy": mass_inside,
        "num_gt_boxes": len(gt_boxes),
        "num_pred_boxes_raw": len(pred_boxes_raw),
        "num_pred_boxes_refined": len(pred_boxes_ref),
        "threshold": DEFAULT_THRESHOLDS[label],
    }
    return pred_boxes_raw, pred_boxes_ref, diag

In [5]:
set_seed(seed)
Path(output_dir).mkdir(parents=True, exist_ok=True)

ann = load_annotations(Path(annotations_csv))
meta = load_meta(Path(meta_csv))
hm_idx = load_heatmap_index(Path(heatmaps_dir))

ann = ann[ann["class_name"].isin(TARGET_LABELS)].copy()

positives = ann[["image_id", "class_name"]].drop_duplicates()
positives = positives.merge(meta, on="image_id", how="left")
positives = positives.merge(hm_idx, on="image_id", how="inner")

segmenter = AnatomySegmenter(device=device)

all_boxes_raw: List[Box] = []
all_boxes_ref: List[Box] = []
all_gt_boxes: List[Box] = []
diag_rows: List[Dict[str, object]] = []
sampled_image_ids: set[str] = set()
image_size_lookup: Dict[str, Tuple[int, int]] = {}

In [6]:
for label in TARGET_LABELS:
    sub = positives[positives["class_name"] == label].copy().drop_duplicates(subset=["image_id"])
    if len(sub) == 0:
        print(f"No cases found for label={label}")
        continue

    n = min(samples_per_label, len(sub))
    sub = sub.sample(n=n, random_state=seed).reset_index(drop=True)
    print(f"Processing {n} sampled positive cases for {label}")

    label_dir = Path(output_dir) / safe_filename(label)
    vis_dir = label_dir / "visualizations"

    for _, row in sub.iterrows():
        image_id = str(row.image_id)
        heatmap_path = str(row.heatmap_path)
        pred_boxes_raw, pred_boxes_ref, diag = process_one_case(
            image_id=image_id,
            label=label,
            heatmap_path=heatmap_path,
            images_root=images_root,
            ann_df=ann,
            segmenter=segmenter,
            save_visuals=True,
            vis_dir=vis_dir,
        )

        sampled_image_ids.add(image_id)
        image_size_lookup[image_id] = (int(diag["width"]), int(diag["height"]))
        diag_rows.append(diag)
        all_boxes_raw.extend(pred_boxes_raw)
        all_boxes_ref.extend(pred_boxes_ref)
        all_gt_boxes.extend(get_gt_boxes(ann, image_id, label))

Processing 75 sampled positive cases for Cardiomegaly
Processing 75 sampled positive cases for Pleural effusion


In [10]:
# Deduplicate GT boxes across labels/images
gt_df = boxes_to_df(all_gt_boxes).drop_duplicates()
all_gt_boxes = [
    Box(r.x_min, r.y_min, r.x_max, r.y_max, 1.0, r.class_name, r.image_id)
    for _, r in gt_df.iterrows()
]

diagnostics_df = pd.DataFrame(diag_rows)
diagnostics_df.to_csv(Path(output_dir) / "diagnostics_all.csv", index=False)

for label in TARGET_LABELS:
    d = diagnostics_df[diagnostics_df["label"] == label]
    if len(d) == 0:
        continue
    summary = {
        "label": label,
        "num_cases": int(len(d)),
        "mean_mass_inside_anatomy": float(d["mass_inside_anatomy"].mean()),
        "median_mass_inside_anatomy": float(d["mass_inside_anatomy"].median()),
        "mean_num_pred_boxes_raw": float(d["num_pred_boxes_raw"].mean()),
        "mean_num_pred_boxes_refined": float(d["num_pred_boxes_refined"].mean()),
    }
    label_dir = Path(output_dir) / safe_filename(label)
    d.to_csv(label_dir / "diagnostics.csv", index=False)
    with open(label_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    print(json.dumps(summary, indent=2))

# Save predictions and GT
boxes_to_df(all_boxes_raw).to_csv(Path(output_dir) / "predictions_raw.csv", index=False)
boxes_to_df(all_boxes_ref).to_csv(Path(output_dir) / "predictions_anatomy_gated.csv", index=False)
boxes_to_df(all_gt_boxes).to_csv(Path(output_dir) / "gt_boxes_sampled.csv", index=False)

{
  "label": "Cardiomegaly",
  "num_cases": 75,
  "mean_mass_inside_anatomy": 0.08381935865925602,
  "median_mass_inside_anatomy": 0.08087279988707344,
  "mean_num_pred_boxes_raw": 6.506666666666667,
  "mean_num_pred_boxes_refined": 1.1333333333333333
}
{
  "label": "Pleural effusion",
  "num_cases": 75,
  "mean_mass_inside_anatomy": 0.18255846563968134,
  "median_mass_inside_anatomy": 0.1803980976221515,
  "mean_num_pred_boxes_raw": 9.813333333333333,
  "mean_num_pred_boxes_refined": 3.12
}


In [12]:
# FROC denominator
num_images = len(sampled_image_ids)

print(f"FROC FP/image denominator uses N={num_images} sampled subset of images")

raw_froc = evaluate_froc(
    labels=TARGET_LABELS,
    predictions=all_boxes_raw,
    gt_boxes=all_gt_boxes,
    num_images=num_images,
    image_size_lookup=image_size_lookup,
    output_dir=Path(output_dir) / "froc_raw",
    prefix="raw",
)
gated_froc = evaluate_froc(
    labels=TARGET_LABELS,
    predictions=all_boxes_ref,
    gt_boxes=all_gt_boxes,
    num_images=num_images,
    image_size_lookup=image_size_lookup,
    output_dir=Path(output_dir) / "froc_anatomy_gated",
    prefix="anatomy_gated",
)

FROC FP/image denominator uses N=145 sampled subset of images
[FROC:raw] Cardiomegaly         GT=  75 Pred=  488 S@0.10=0.0133 S@0.25=0.0667 S@0.50=0.1467
[FROC:raw] Pleural effusion     GT=  93 Pred=  736 S@0.10=0.1075 S@0.25=0.1505 S@0.50=0.2366
[FROC:anatomy_gated] Cardiomegaly         GT=  75 Pred=   85 S@0.10=0.6933 S@0.25=0.7733 S@0.50=0.7733
[FROC:anatomy_gated] Pleural effusion     GT=  93 Pred=  234 S@0.10=0.1183 S@0.25=0.2366 S@0.50=0.3656


In [15]:
# Side-by-side comparison
compare = raw_froc.merge(gated_froc, on="label", suffixes=("_raw", "_gated"))
compare.to_csv(Path(output_dir) / "froc_comparison.csv", index=False)

# Simple comparison summary
compare_summary_rows = []
for label in TARGET_LABELS:
    d = diagnostics_df[diagnostics_df["label"] == label]
    if len(d) == 0:
        continue
    compare_summary_rows.append({
        "label": label,
        "num_cases": len(d),
        "mean_mass_inside_anatomy": d["mass_inside_anatomy"].mean(),
        "mean_num_pred_boxes_raw": d["num_pred_boxes_raw"].mean(),
        "mean_num_pred_boxes_refined": d["num_pred_boxes_refined"].mean(),
    })
pd.DataFrame(compare_summary_rows).to_csv(Path(output_dir) / "compare_summary.csv", index=False)

print("Done. Main outputs:")
print(f"- {Path(output_dir) / 'diagnostics_all.csv'}")
print(f"- {Path(output_dir) / 'predictions_raw.csv'}")
print(f"- {Path(output_dir) / 'predictions_anatomy_gated.csv'}")
print(f"- {Path(output_dir) / 'froc_raw' / 'raw_froc_summary.csv'}")
print(f"- {Path(output_dir) / 'froc_anatomy_gated' / 'anatomy_gated_froc_summary.csv'}")
print(f"- {Path(output_dir) / 'froc_comparison.csv'}")

Done. Main outputs:
- /home/woody/iwi5/iwi5362h/ALBEF/results/clarity_end_to_end/diagnostics_all.csv
- /home/woody/iwi5/iwi5362h/ALBEF/results/clarity_end_to_end/predictions_raw.csv
- /home/woody/iwi5/iwi5362h/ALBEF/results/clarity_end_to_end/predictions_anatomy_gated.csv
- /home/woody/iwi5/iwi5362h/ALBEF/results/clarity_end_to_end/froc_raw/raw_froc_summary.csv
- /home/woody/iwi5/iwi5362h/ALBEF/results/clarity_end_to_end/froc_anatomy_gated/anatomy_gated_froc_summary.csv
- /home/woody/iwi5/iwi5362h/ALBEF/results/clarity_end_to_end/froc_comparison.csv
